In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

from scipy.stats import median_abs_deviation

In [2]:
import sys
sys.path.append('/g/data/rd53/wy2165/disequilibrium/code_analysis/')
from Lowess_fit import run_lowess_from_csv

In [3]:
region = 19
#lowess_y_cols = ['AAR_mean_area_weighted_mean',
#                 'disequilibrium_area_weighted_mean']
lowess_y_cols = ['AAR_mean_median',
                 'disequilibrium_median']

stats_vars = ['AAR_mean', 'AAR_steady', 'disequilibrium']

outpath = Path('/g/data/rd53/wy2165/disequilibrium/pygem_oggm/')

In [4]:
tag = 'median_a'
ds = xr.open_dataset(outpath / f'PyGEM_global_glacier_stats_{tag}.nc')

In [5]:
missing =  pd.read_csv(f'/scratch/k10/wy2165/PyGEM/frontalablation_data/analysis/{region}-frontalablation_cal_ind-missing.csv')
ind = pd.read_csv(f'/scratch/k10/wy2165/PyGEM/frontalablation_data/analysis/{region}-frontalablation_cal_ind.csv')

### Stats

In [6]:
def xr_nanmad(da):
    return xr.apply_ufunc(
        median_abs_deviation,
        da,
        input_core_dims=[['rgi_id']],
        output_core_dims=[[]],
        vectorize=True,
        dask='parallelized',
        kwargs={'nan_policy': 'omit'},
        output_dtypes=[float],
    )


def area_weighted_mean(da, area):
    valid_area = area.where(np.isfinite(da))
    return (da * valid_area).sum(dim='rgi_id', skipna=True) / valid_area.sum(
        dim='rgi_id',
        skipna=True,
    )


def summarize_stats_for_class(ds, class_name):
    area = ds['rgi_area_km2']

    summary = xr.Dataset()

    for var in stats_vars:
        da = ds[var]
        summary[f'{var}_mean'] = da.mean(dim='rgi_id', skipna=True)
        summary[f'{var}_std'] = da.std(dim='rgi_id', skipna=True)
        summary[f'{var}_median'] = da.median(dim='rgi_id', skipna=True)
        summary[f'{var}_MAD'] = xr_nanmad(da)
        summary[f'{var}_area_weighted_mean'] = area_weighted_mean(da, area)

    df = summary.to_dataframe().reset_index()

    df['region_class'] = class_name
    df['n_glaciers'] = int(area.notnull().sum().item())
    df['rgi_area_km2'] = float(area.sum(skipna=True).item())

    cols = [
        'region_class',
        'n_glaciers',
        'rgi_area_km2',
        'experiment',
        'gcm',
        'period_scenario',
        'temp_ch_ipcc',
    ]

    stat_cols = []
    for var in stats_vars:
        stat_cols.extend([
            f'{var}_mean',
            f'{var}_std',
            f'{var}_median',
            f'{var}_MAD',
            f'{var}_area_weighted_mean',
        ])

    return df[cols + stat_cols]


def save_mt_stats_csv(ds, class_name, outpath, tag):
    if ds.sizes.get('rgi_id', 0) == 0:
        print('Skipped empty class:', class_name)
        return

    df = summarize_stats_for_class(ds, class_name)
    if df.empty:
        print('Skipped empty class:', class_name)
        return

    output_csv = outpath / f'PyGEM_glacier_stats_{class_name}{tag}.csv'
    df.to_csv(output_csv, index=False)
    print('Saved:', output_csv)

#### Marine-terminating glaciers with frontal ablation observations

In [7]:
class_name = 'MT_19_with_obs'
mask = (
    (ds['region'] == 19)
    & (ds['is_tidewater'] == 1)
    & (ds['rgi_id'].isin(ind['RGIId']))
)
ds_sub = ds.where(mask, drop=True)

MT_class = np.repeat(class_name, ds_sub.sizes['rgi_id'])

ds_sub = ds_sub.assign_coords(region_class=('rgi_id', MT_class))

In [8]:
save_mt_stats_csv(ds_sub, class_name, outpath, '')

Saved: /g/data/rd53/wy2165/disequilibrium/pygem_oggm/PyGEM_glacier_stats_MT_19_with_obs.csv


In [9]:
for y_col in lowess_y_cols:
    run_lowess_from_csv(
        outpath / f'PyGEM_glacier_stats_{class_name}.csv',
        outpath / f'PyGEM_glacier_stats_{class_name}_{y_col}_lowess_fit.csv',
        trials_output_csv=None,
        x_col='temp_ch_ipcc',
        y_col=y_col,
        qs=None,
        preliminary_num_fits=500,
        final_num_fits=2000,
        robust_iters=2,
    )

#### Marine-terminating glaciers missing frontal ablation observations
#### median_a

In [10]:
class_name = 'MT_19_missing_obs'
mask = (
    (ds['region'] == 19)
    & (ds['is_tidewater'] == 1)
    & (ds['rgi_id'].isin(missing['RGIId']))
)
ds_sub = ds.where(mask, drop=True)

MT_class = np.repeat(class_name, ds_sub.sizes['rgi_id'])

ds_sub = ds_sub.assign_coords(region_class=('rgi_id', MT_class))

In [11]:
save_mt_stats_csv(ds_sub, class_name, outpath, f'_{tag}')

Saved: /g/data/rd53/wy2165/disequilibrium/pygem_oggm/PyGEM_glacier_stats_MT_19_missing_obs_median_a.csv


In [12]:
for y_col in lowess_y_cols:
    run_lowess_from_csv(
        outpath / f'PyGEM_glacier_stats_{class_name}_{tag}.csv',
        outpath / f'PyGEM_glacier_stats_{class_name}_{y_col}_{tag}_lowess_fit.csv',
        trials_output_csv=None,
        x_col='temp_ch_ipcc',
        y_col=y_col,
        qs=None,
        preliminary_num_fits=500,
        final_num_fits=2000,
        robust_iters=2,
    )

#### Marine-terminating glaciers missing frontal ablation observations
#### median_k

In [13]:
tag = 'median_k'
ds = xr.open_dataset(outpath / f'PyGEM_global_glacier_stats_{tag}.nc')

In [14]:
class_name = 'MT_19_missing_obs'
mask = (
    (ds['region'] == 19)
    & (ds['is_tidewater'] == 1)
    & (ds['rgi_id'].isin(missing['RGIId']))
)
ds_sub = ds.where(mask, drop=True)

MT_class = np.repeat(class_name, ds_sub.sizes['rgi_id'])

ds_sub = ds_sub.assign_coords(region_class=('rgi_id', MT_class))

In [15]:
save_mt_stats_csv(ds_sub, class_name, outpath, f'_{tag}')

Saved: /g/data/rd53/wy2165/disequilibrium/pygem_oggm/PyGEM_glacier_stats_MT_19_missing_obs_median_k.csv


In [16]:
for y_col in lowess_y_cols:
    run_lowess_from_csv(
        outpath / f'PyGEM_glacier_stats_{class_name}_{tag}.csv',
        outpath / f'PyGEM_glacier_stats_{class_name}_{y_col}_{tag}_lowess_fit.csv',
        trials_output_csv=None,
        x_col='temp_ch_ipcc',
        y_col=y_col,
        qs=None,
        preliminary_num_fits=500,
        final_num_fits=2000,
        robust_iters=2,
    )